# NB03 — Memory-bounded mined normalization

Stage S3. This notebook mines token substitutions and dropped-token statistics
from labelled training pairs only. It excludes the persisted S1 holdout before
reading positives, so the holdout remains an honest evaluation set.

The implementation streams DuckDB result batches and never loads all 12M source
rows into a pandas DataFrame. It is intended for the available 8 GiB environment.

In [ ]:
import json, os, pathlib, sys, time
import pandas as pd

ON_SAGEMAKER = pathlib.Path('/opt/ml').exists() or 'SM_CURRENT_HOST' in os.environ
ROOT = pathlib.Path(os.environ.get(
    'BER_ROOT',
    '/home/ec2-user/SageMaker/ber' if ON_SAGEMAKER else '/Users/pocketfm/Desktop/amazon_ML/Working Folder'
))
DATA = ROOT / 'student_resource/dataset'
ART = ROOT / 'artifacts'
ART.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / 'src'))
from business_entity_resolution.normalization_mined import NormalizationMiner, save_mined_tables
from business_entity_resolution.normalize import joint_key, tokenize
print(f'ROOT={ROOT}')

## Stream positive pairs

DuckDB performs the joins over the TSV files and yields fixed-size batches. The
holdout S1 IDs are excluded in SQL before the miner sees any pair.

In [ ]:
import duckdb

def sql_path(path):
    return str(path).replace('\\', '\\\\').replace("'", "''")

holdout = pd.read_parquet(ART / 'holdout_s1.parquet', columns=['entity_id'])
holdout.to_parquet(ART / 'nb03_holdout_ids.parquet', index=False)
con = duckdb.connect(database=str(ART / 'nb03.duckdb'))
con.execute("PRAGMA threads=2")
con.execute("PRAGMA memory_limit='5GB'")
(ART / 'nb03_tmp').mkdir(parents=True, exist_ok=True)
con.execute("SET temp_directory = '" + sql_path(ART / 'nb03_tmp') + "'")
con.execute("CREATE OR REPLACE TEMP TABLE holdout_ids AS SELECT * FROM read_parquet(?)", [str(ART / 'nb03_holdout_ids.parquet')])

s1 = sql_path(DATA / 'train/train_source1.tsv')
s2 = sql_path(DATA / 'train/train_source2.tsv')
s3 = sql_path(DATA / 'train/train_source3.tsv')
gt = sql_path(DATA / 'train/train_ground_truth.tsv')
source_cols = "columns={entity_id:'VARCHAR', business_name:'VARCHAR', business_address:'VARCHAR', country:'VARCHAR'}"
gt_cols = "columns={source1_entity_id:'VARCHAR', matched_entity_ids:'VARCHAR'}"
mining_limit = int(os.environ.get('BER_S3_MAX_PAIRS', '1000000'))
base_query = f"""
WITH positives AS (
  SELECT source1_entity_id,
         unnest(string_split(coalesce(matched_entity_ids, ''), ',')) AS matched_id
  FROM read_csv('{gt}', delim='\\t', header=true, auto_detect=false, {gt_cols})
  WHERE matched_entity_ids IS NOT NULL AND matched_entity_ids <> ''
    AND source1_entity_id NOT IN (SELECT entity_id FROM holdout_ids)
), source2 AS (
  SELECT * FROM read_csv('{s2}', delim='\\t', header=true, auto_detect=false, {source_cols})
), source3 AS (
  SELECT * FROM read_csv('{s3}', delim='\\t', header=true, auto_detect=false, {source_cols})
), refs AS (
  SELECT * FROM read_csv('{s1}', delim='\\t', header=true, auto_detect=false, {source_cols})
)
SELECT r.business_name AS name_left, r.business_address AS address_left,
       x.business_name AS name_right, x.business_address AS address_right
FROM positives p JOIN refs r ON r.entity_id=p.source1_entity_id
JOIN source2 x ON x.entity_id=p.matched_id
UNION ALL
SELECT r.business_name, r.business_address, x.business_name, x.business_address
FROM positives p JOIN refs r ON r.entity_id=p.source1_entity_id
JOIN source3 x ON x.entity_id=p.matched_id
"""
query = base_query + f'\nLIMIT {mining_limit}'


In [ ]:
# Exact transformations are the S3-v1 baseline. Fuzzy edit-distance mining is
# intentionally deferred: it is too expensive across 7M pairs on this 2-vCPU host.
miner = NormalizationMiner(enable_fuzzy=False)
n_pairs = 0
t0 = time.time()
reader = con.execute(query).fetch_record_batch(rows_per_batch=25_000)
for batch in reader:
    df = batch.to_pandas()
    miner.update(df.itertuples(index=False, name=None))
    n_pairs += len(df)
    del df
print(f'positive pairs mined={n_pairs:,} elapsed={time.time()-t0:.1f}s')
# Conservative default: corrected conditional rates make support=3 too noisy for
# a precision-weighted metric. Sweep lower support only after this v1 audit.
tables = miner.finish(
    min_support=10, min_confidence=0.90, min_drop_rate=0.80,
    min_drop_support=1000,
)
save_mined_tables(tables, ART / 'normalization_v1.json')
summary = {
    'stage': 'S3-v6-conservative-drops',
    'pairs_mined': n_pairs,
    'mining_pair_limit': mining_limit,
    'holdout_excluded': len(holdout),
    'name_aliases': len(tables.name_aliases),
    'address_aliases': len(tables.address_aliases),
    'name_drop_tokens': len(tables.name_drop_tokens),
    'address_drop_tokens': len(tables.address_drop_tokens),
    'config': dict(tables.config),
    'mining_diagnostics': miner.stats_summary(),
}
# Evaluate pair-level coverage on the untouched holdout.  This is a diagnostic,
# not a leaderboard prediction: it deliberately does not mine from holdout rows.
evaluation_query = base_query.replace(
    "    AND source1_entity_id NOT IN (SELECT entity_id FROM holdout_ids)\n",
    "    AND source1_entity_id IN (SELECT entity_id FROM holdout_ids)\n",
)
coverage = {
    'positive_pairs': 0,
    'v0_joint_equal': 0,
    'v1_joint_equal': 0,
    'name_zero_overlap_pairs': 0,
    'v0_zero_overlap_equal': 0,
    'v1_zero_overlap_equal': 0,
}
reader = con.execute(evaluation_query).fetch_record_batch(rows_per_batch=25_000)
for batch in reader:
    df = batch.to_pandas()
    for name_left, address_left, name_right, address_right in df.itertuples(index=False, name=None):
        coverage['positive_pairs'] += 1
        v0_equal = joint_key(name_left, address_left) == joint_key(name_right, address_right)
        v1_equal = tables.joint_key(name_left, address_left) == tables.joint_key(name_right, address_right)
        coverage['v0_joint_equal'] += int(v0_equal)
        coverage['v1_joint_equal'] += int(v1_equal)
        left_name = set(tokenize(name_left))
        right_name = set(tokenize(name_right))
        zero_overlap = not (left_name & right_name)
        coverage['name_zero_overlap_pairs'] += int(zero_overlap)
        coverage['v0_zero_overlap_equal'] += int(zero_overlap and v0_equal)
        coverage['v1_zero_overlap_equal'] += int(zero_overlap and v1_equal)
    del df
coverage['v0_pair_recall'] = coverage['v0_joint_equal'] / max(1, coverage['positive_pairs'])
coverage['v1_pair_recall'] = coverage['v1_joint_equal'] / max(1, coverage['positive_pairs'])
coverage['zero_overlap_v0_recall'] = coverage['v0_zero_overlap_equal'] / max(1, coverage['name_zero_overlap_pairs'])
coverage['zero_overlap_v1_recall'] = coverage['v1_zero_overlap_equal'] / max(1, coverage['name_zero_overlap_pairs'])
summary['holdout_pair_coverage'] = coverage
summary_path = ART / 'normalization_v1_summary.json'
summary_path.write_text(json.dumps(summary, indent=2) + '\n')
run_record = {
    'ts': time.strftime('%Y-%m-%dT%H:%M:%S'),
    'stage': summary['stage'],
    'notebook': 'NB03_normalization',
    'config': summary['config'],
    'metrics': {
        'pairs_mined': n_pairs,
        'name_aliases': summary['name_aliases'],
        'address_aliases': summary['address_aliases'],
        'v0_pair_recall': coverage['v0_pair_recall'],
        'v1_pair_recall': coverage['v1_pair_recall'],
    },
    'notes': 'Holdout excluded from mining; coverage is pair-level diagnostic only.',
}
with (ART / 'runs.jsonl').open('a', encoding='utf-8') as f:
    f.write(json.dumps(run_record, sort_keys=True) + '\n')
print(json.dumps(summary, indent=2))

## Exit checks

The first run must report the number of mined positives and table sizes. Before
using these tables for matching, compare v0 and v1 on the fixed holdout and
inspect hard-tail coverage and collision counts. No leaderboard upload is made
by this notebook.